# Deep CoxPH Training on AIDS Classification Dataset

This notebook demonstrates how to train a `DeepCoxPH` model from the `auton-survival` library using the `AIDS_Classification_50000.csv` dataset.

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from auton_survival import DeepCoxPH
from auton_survival.metrics import survival_regression_metric

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Data

In [33]:
df = pd.read_csv('AIDS_Classification_50000.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (50000, 23)


,time,trt,age,wtkg,hemo,homo,drugs,karnof,oprior,z30,...,str2,strat,symptom,treat,offtrt,cd40,cd420,cd80,cd820,infected
0,1073,1,37,79.46339,0,1,0,100,0,1,...,1,2,0,1,0,322,469,882,754,1
1,324,0,33,73.02314,0,1,0,90,0,1,...,1,3,1,1,1,168,575,1035,1525,1
2,495,1,43,69.47793,0,1,0,100,0,1,...,1,1,0,0,0,377,333,1147,1088,1
3,1201,3,42,89.15934,0,1,0,100,1,1,...,1,3,0,0,0,238,324,775,1019,1
4,934,0,37,137.46581,0,1,0,100,0,0,...,0,3,0,0,1,500,443,1601,849,0


## 2. Preprocessing

We separate the data into features (`x`), time (`t`), and event (`e`). 
Numerical features are standardized, and categorical features are one-hot encoded.

In [34]:
# Define columns
time_col = 'time'
event_col = 'infected'



x = df.drop(columns=[time_col, event_col])
t = df[time_col].values
e = df[event_col].values

cat_cols = [col for col in x.columns if x[col].nunique() < 10]
num_cols = [col for col in x.columns if col not in cat_cols]

# Check for missing values
print("Missing values:\n", df.isnull().sum().sum())

Missing values:
 0


In [35]:
# Compute horizons
horizons = [0.25, 0.5, 0.75]
times = np.quantile(t[e == 1], horizons).tolist()
print(times)

[496.0, 992.0, 1127.0]


In [36]:
# train-test and validation split
x_train, x_temp, t_train, t_temp, e_train, e_temp = train_test_split(
    x, t, e, test_size=0.30, random_state=42, stratify=e
)
x_test, x_val, t_test, t_val, e_test, e_val  = train_test_split(
    x_temp, t_temp, e_temp, test_size=0.30, random_state=42, stratify=e_temp
)
# Remove max values from test/val (often causes issues in evaluation if they exceed train max)
mask_test = t_test < t_train.max()
x_test = x_test[mask_test]
t_test = t_test[mask_test]
e_test = e_test[mask_test]

mask_val = t_val < t_train.max()
x_val = x_val[mask_val]
t_val = t_val[mask_val]
e_val = e_val[mask_val]

# Prepare sksurv format targets
y_train_surv = Surv.from_arrays(e_train.astype(bool), t_train)
y_test_surv = Surv.from_arrays(e_test.astype(bool), t_test)
y_val_surv = Surv.from_arrays(e_val.astype(bool), t_val)


# print final shape of train, test and validation set
print(f"Train shape: {x_train.shape}")
print(f"Test shape: {x_test.shape}")
print(f"Validation shape: {x_val.shape}")

Train shape: (35000, 21)
Test shape: (10480, 21)
Validation shape: (4495, 21)


In [37]:
# Column transformer for preprocessing with OneHotEncoder and StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# Create the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        (
            "cat",
            OneHotEncoder(drop="first", handle_unknown="ignore"),
            cat_cols,
        ),
    ],
    remainder="passthrough",  # Keep any other columns unchanged (optional)
)

In [38]:
# Feature scaling

x_train = preprocessor.fit_transform(x_train)
x_test = preprocessor.transform(x_test)
x_val = preprocessor.transform(x_val)

## 3. Train DeepCoxPH Model

We conduct a hyperparameter search to find the best model.

In [39]:
# from sklearn.model_selection import ParameterGrid
# from sksurv.metrics import concordance_index_ipcw

# # Define the hyperparameter grid
# param_grid = {
#     'layers': [[32,32], [64, 64], [100], [100, 100]],
#     'learning_rate': [1e-3, 1e-4, 5e-4, 1e-5],
#     'optimizer': ['Adam', 'RMSProp', 'SGD']
# }
# parameters = ParameterGrid(param_grid)

# models = []
# y_train = np.array(
#     [(e_train[i], t_train[i]) for i in range(len(e_train))],
#     dtype=[("e", bool), ("t", float)],
# )
# y_val = np.array(
#     [(e_val[i], t_val[i]) for i in range(len(e_val))],
#     dtype=[("e", bool), ("t", float)],
# )
# print("Starting Hyperparameter Tuning...")
# for params in parameters:
#     print(f"\nTesting params: {params}")
#     # Initialize model with current layers
#     curr_model = DeepCoxPH(layers=params['layers'])
    
#     # Fit the model with current learning_rate and optimizer
#     curr_model.fit(x_train, t_train, e_train, 
#                 #   val_data=(x_val, t_val, e_val),
#                   batch_size=64, 
#                   iters=25, # Using 25 iterations for tuning speed
#                   learning_rate=params['learning_rate'],
#                   optimizer=params['optimizer'])
#     out_risk = curr_model.predict_risk(x_val, t=times[1])
#     c_index = concordance_index_ipcw(y_train, y_val, out_risk[:, 0], times[1])[0]
#     print(f"C-index: {c_index:.4f}")
#     models.append([[c_index, curr_model, params]])

# best_model = max(models)
# print("Best model params:", best_model[0][2])
# model = best_model[0][1]

Best model params: {'layers': [64, 64], 'learning_rate': 0.0001, 'optimizer': 'RMSProp'}

In [40]:
# Train the model
from auton_survival import DeepCoxPH
model = DeepCoxPH(layers=[64, 64])
model.fit(x_train, t_train, e_train, val_data=(x_val, t_val, e_val), 
         learning_rate=1e-4, optimizer='RMSProp', iters=100)

 37%|███▋      | 37/100 [00:27<00:46,  1.36it/s]


In [41]:
# Predict risk and survival probabilities
out_risk = model.predict_risk(x_test, t=times)
out_survival = model.predict_survival(x_test, t=times)

In [42]:
# auton-survival (DeepCoxPH, DSM)
risk_at_times = model.predict_risk(x_test, t=times)
surv_probs = model.predict_survival(x_test, t=times)

# For global C-index, approximate global risk (e.g. mean risk over horizons or expected survival)
# Using mean risk over the 3 horizons for simplicity and consistency with previous notes
global_risk = np.mean(risk_at_times, axis=1)

## 4. Evaluation

We estimate the risk and calculate the C-index (Concordance Index) on the test set.

In [43]:
results_quartile = []
results_global = []
for i, t_val in enumerate(times):
        # Time-dependent C-index
        # CoxPH has proportional hazards -> single risk score valid for all times
        # But specific SkSurv c-index expects one risk score.
        # Ideally use the risk_at_time for TD-Cindex if available, or the global risk.
        # Using global risk is standard for PH models, but let's use the time-specific risk surface if we calculated it (1-Surv).

        
        current_risk = risk_at_times[:, i]

        ctd = concordance_index_ipcw(y_train_surv, y_test_surv, current_risk, t_val)[0]
        
        # Brier Score
        # brier_score returns (times, scores), we want the score at index i
        # But we computed surv_probs[:, i] manually for Cox.
        # Let's use the bulk brier function for consistency
        # For specific time t_val:
        bs = brier_score(y_train_surv, y_test_surv, surv_probs[:, i].reshape(-1, 1), [t_val])[1][0]
        
        # AUC
        # Cumulative Dynamic AUC returns (aucs, mean_auc)
        # We need the AUC at this specific time t_val.
        # We can pass just this one time point.
        auc_val_tuple = cumulative_dynamic_auc(y_train_surv, y_test_surv, current_risk, t_val)
        auc = auc_val_tuple[0][0]
        
        results_quartile.append({
            "Quartile_Time": t_val,
            "Quartile_Idx": f"Q{i+1}",
            "C-Index (TD)": ctd,
            "Brier Score": bs,
            "AUC Score": auc
        })

# --- Global Metrics ---
# Integrated Brier / AUC
ibs = integrated_brier_score(y_train_surv, y_test_surv, surv_probs, times)

# Integrated AUC
_, i_auc = cumulative_dynamic_auc(y_train_surv, y_test_surv, risk_at_times, times)

# Global Concordance (Traditional C-index for censored data)
# Note: For time-varying risk models, this is an approximation using a representative risk score
c_global = concordance_index_censored(y_test_surv["event"], y_test_surv["time"], global_risk)[0]

results_global.append({
    "Integrated Brier Score": ibs,
    "Integrated ROC-AUC": i_auc,
    "Global C-Index": c_global
})

df_quartile = pd.DataFrame(results_quartile)
df_global = pd.DataFrame(results_global)

In [44]:
display(df_quartile)
display(df_global)


,Quartile_Time,Quartile_Idx,C-Index (TD),Brier Score,AUC Score
0,496.0,Q1,0.704660,0.070796,0.713199
1,992.0,Q2,0.686319,0.139188,0.700052
2,1127.0,Q3,0.660347,0.204536,0.663016


,Integrated Brier Score,Integrated ROC-AUC,Global C-Index
0,0.119298,0.687283,0.672805
